In [4]:
import pandas as pd
import numpy as np
from geopy.distance import distance
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Load dataset
data = pd.read_csv('uber.csv')

# Drop unnecessary columns
data = data.drop(columns=['Unnamed: 0', 'key'])

# Check for null values and drop rows with any null values
data = data.dropna()

# Drop duplicate rows, if any
data = data.drop_duplicates()

# Filter out rows with invalid latitude and longitude values
data = data[(data.pickup_latitude < 90) & (data.dropoff_latitude < 90) &
            (data.pickup_latitude > -90) & (data.dropoff_latitude > -90) &
            (data.pickup_longitude < 180) & (data.dropoff_longitude < 180) &
            (data.pickup_longitude > -180) & (data.dropoff_longitude > -180)]

# Convert 'pickup_datetime' to datetime format and extract time-based features
data['pickup_datetime'] = pd.to_datetime(data['pickup_datetime'], errors='coerce')
data['pickup_hour'] = data['pickup_datetime'].dt.hour
data['pickup_day'] = data['pickup_datetime'].dt.day
data['pickup_month'] = data['pickup_datetime'].dt.month
data['pickup_dayofweek'] = data['pickup_datetime'].dt.dayofweek

# Calculate distance using geopy
data['distance_miles'] = [
    round(distance((data.pickup_latitude[i], data.pickup_longitude[i]),
                   (data.dropoff_latitude[i], data.dropoff_longitude[i])).miles, 2)
    for i in data.index
]

# Prepare the data for regression
data = data.drop(columns=['pickup_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude'])
data = data[data['fare_amount'] > 0]
data = data[data['distance_miles'] > 0]

# Define features and target variable
X = data[['distance_miles', 'pickup_hour', 'pickup_day', 'pickup_month', 'pickup_dayofweek', 'passenger_count']]
y = data['fare_amount']

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the scaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform both training and test data
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize the model
model = LinearRegression()

# Train the model on the scaled training data
model.fit(X_train, y_train)

# Make predictions on the scaled test data
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# Calculate R-squared for train and test sets
train_r2 = model.score(X_train, y_train)
test_r2 = model.score(X_test, y_test)

# Print results
print("Mean Absolute Error (MAE):", mae)
print("Root Mean Squared Error (RMSE):", rmse)
print("Train R-squared (R²):", train_r2)
print("Test R-squared (R²):", test_r2)


Mean Absolute Error (MAE): 5.905289557881476
Root Mean Squared Error (RMSE): 9.384974483657857
Train R-squared (R²): 0.0018856733550522975
Test R-squared (R²): 0.0017674565323515523


In [5]:
import pandas as pd
import numpy as np
from geopy.distance import distance
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load dataset
data = pd.read_csv('uber.csv')

# Drop unnecessary columns
data = data.drop(columns=['Unnamed: 0', 'key'])

# Check for null values and drop rows with any null values
data = data.dropna()

# Drop duplicate rows, if any
data = data.drop_duplicates()

# Filter out rows with invalid latitude and longitude values
data = data[(data.pickup_latitude < 90) & (data.dropoff_latitude < 90) &
            (data.pickup_latitude > -90) & (data.dropoff_latitude > -90) &
            (data.pickup_longitude < 180) & (data.dropoff_longitude < 180) &
            (data.pickup_longitude > -180) & (data.dropoff_longitude > -180)]

# Convert 'pickup_datetime' to datetime format and extract time-based features
data['pickup_datetime'] = pd.to_datetime(data['pickup_datetime'], errors='coerce')
data['pickup_hour'] = data['pickup_datetime'].dt.hour
data['pickup_day'] = data['pickup_datetime'].dt.day
data['pickup_month'] = data['pickup_datetime'].dt.month
data['pickup_dayofweek'] = data['pickup_datetime'].dt.dayofweek

# Calculate distance using geopy
data['distance_miles'] = [
    round(distance((data.pickup_latitude[i], data.pickup_longitude[i]),
                   (data.dropoff_latitude[i], data.dropoff_longitude[i])).miles, 2)
    for i in data.index
]

# Prepare the data for regression
data = data.drop(columns=['pickup_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude'])
data = data[data['fare_amount'] > 0]
data = data[data['distance_miles'] > 0]

# Define features and target variable
X = data[['distance_miles', 'pickup_hour', 'pickup_day', 'pickup_month', 'pickup_dayofweek', 'passenger_count']]
y = data['fare_amount']

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the scaler and scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Loop through different degrees and evaluate the model
for degree in range(1, 6):  # Test polynomial degrees from 1 to 5
    # Apply polynomial transformation
    poly = PolynomialFeatures(degree=degree)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)

    # Initialize the model
    poly_model = LinearRegression()

    # Train the model on the polynomial-transformed training data
    poly_model.fit(X_train_poly, y_train)

    # Make predictions on the polynomial-transformed test data
    y_test_pred_poly = poly_model.predict(X_test_poly)
    y_train_pred_poly = poly_model.predict(X_train_poly)

    # Evaluate the model on the test data
    mae_poly = mean_absolute_error(y_test, y_test_pred_poly)
    rmse_poly = np.sqrt(mean_squared_error(y_test, y_test_pred_poly))
    test_r2_poly = r2_score(y_test, y_test_pred_poly)
    train_r2_poly = r2_score(y_train, y_train_pred_poly)

    # Print results for each degree
    print(f"\nPolynomial Regression (Degree {degree}) Results:")
    print("Mean Absolute Error (MAE):", mae_poly)
    print("Root Mean Squared Error (RMSE):", rmse_poly)
    print("Train R-squared (R²):", train_r2_poly)
    print("Test R-squared (R²):", test_r2_poly)



Polynomial Regression (Degree 1) Results:
Mean Absolute Error (MAE): 5.905289557881476
Root Mean Squared Error (RMSE): 9.384974483657857
Train R-squared (R²): 0.0018856733550522975
Test R-squared (R²): 0.0017674565323515523

Polynomial Regression (Degree 2) Results:
Mean Absolute Error (MAE): 5.866761672987475
Root Mean Squared Error (RMSE): 9.383297844410041
Train R-squared (R²): 0.011475801064630864
Test R-squared (R²): 0.0021240960440377377

Polynomial Regression (Degree 3) Results:
Mean Absolute Error (MAE): 4.748821704729526
Root Mean Squared Error (RMSE): 8.803166711703305
Train R-squared (R²): 0.20630350161545175
Test R-squared (R²): 0.12169898463265871

Polynomial Regression (Degree 4) Results:
Mean Absolute Error (MAE): 3.5296273079871923
Root Mean Squared Error (RMSE): 9.888628718056623
Train R-squared (R²): 0.4548642606845731
Test R-squared (R²): -0.10824983146470801

Polynomial Regression (Degree 5) Results:
Mean Absolute Error (MAE): 2.9811918641758695
Root Mean Squared E